# Day 18 — MLflow Tracking with a SQLite Database

MLflow runs as a local server. Everything it records is saved **inside this `Day 18` folder**:

| File / folder | What it holds |
|---|---|
| `mlflow.db` | SQLite database — experiments, runs, parameters, metrics, tags |
| `mlartifacts/` | Saved models and other artifact files |
| `mlflow_server.log` | The server's log output (check it if the server does not start) |

**Order:** install → import → start the server → connect → prepare the Iris data → train and log a run (with the `Training Info` tag) → view it in the UI at http://127.0.0.1:5000 → load the model back and predict → stop the server when finished.

## 1. Install MLflow

`%pip` installs into the **same Python the notebook kernel uses**. `!pip` can install into a different Python, which causes `ModuleNotFoundError: No module named 'mlflow'`.
After installing for the first time, **restart the kernel** before running the next cell.

In [8]:
%pip install mlflow

## 2. Imports

In [9]:
import os
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

import mlflow
import pandas as pd
from mlflow.models import infer_signature
from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split


## 3. Start the MLflow server (SQLite backend)

- `--backend-store-uri sqlite:///…/mlflow.db` → runs, params and metrics go into the SQLite file.
- `--artifacts-destination …/mlartifacts` → models and other files go into this folder.

The server runs in the background, so the notebook can keep working. If it is already running (for example after a kernel restart), this cell reuses it instead of starting a second one.

This cell also **pins the working directory to the Day 18 folder** and sets `MLFLOW_TRACKING_URI`, so nothing (neither our code nor MLflow) leaks files such as `./mlruns` into the folder the kernel happened to be launched from.

In [10]:
# ---------------------------------------------------------------------------
# Work out where "Day 18" is, no matter how the kernel was launched.
# Order: VS Code notebook path -> __file__ -> nearest folder with a notebook
#        -> current working directory.
# ---------------------------------------------------------------------------
def _find_notebook_dir() -> Path:
    for key in ("__vsc_ipynb_file__", "__file__"):
        value = globals().get(key)
        if value:
            p = Path(value).expanduser()
            if p.exists():
                return p.resolve().parent

    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        try:
            if any(candidate.glob("*.ipynb")):
                return candidate
        except OSError:
            pass
    return cwd


BASE_DIR = _find_notebook_dir()
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Pin the working directory. From here on, every relative path (ours,
# MLflow's, the server subprocess's) resolves inside Day 18.
os.chdir(BASE_DIR)

DB_PATH = BASE_DIR / "mlflow.db"
ARTIFACT_DIR = BASE_DIR / "mlartifacts"
LOG_PATH = BASE_DIR / "mlflow_server.log"

HOST, PORT = "127.0.0.1", 5000
TRACKING_URI = f"http://{HOST}:{PORT}"

# Stop MLflow from ever creating the default ./mlruns folder in cwd.
os.environ["MLFLOW_TRACKING_URI"] = TRACKING_URI


def server_is_up() -> bool:
    try:
        with urllib.request.urlopen(f"{TRACKING_URI}/health", timeout=1) as r:
            return r.status == 200
    except OSError:
        return False


if server_is_up():
    print("MLflow server is already running at", TRACKING_URI)
else:
    # Give the child a clean env so MLFLOW_TRACKING_URI cannot fight the
    # explicit --backend-store-uri / --artifacts-destination flags.
    server_env = os.environ.copy()
    server_env.pop("MLFLOW_TRACKING_URI", None)

    server_log = open(LOG_PATH, "w")
    server = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "mlflow",
            "server",
            "--backend-store-uri",
            f"sqlite:///{DB_PATH.as_posix()}",
            "--artifacts-destination",
            ARTIFACT_DIR.as_posix(),
            "--host",
            HOST,
            "--port",
            str(PORT),
        ],
        cwd=BASE_DIR,
        env=server_env,
        stdout=server_log,
        stderr=subprocess.STDOUT,
        creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),
    )

    # The first start creates the database tables, so give it up to 60 seconds
    for _ in range(60):
        if server_is_up() or server.poll() is not None:
            break
        time.sleep(1)

    if server_is_up():
        print("MLflow server started at", TRACKING_URI)
    else:
        print("MLflow server did not start - check", LOG_PATH)

print("Base dir :", BASE_DIR)
print("Database :", DB_PATH)
print("Artifacts:", ARTIFACT_DIR)
print("Log      :", LOG_PATH)

MLflow server is already running at http://127.0.0.1:5000
Base dir : C:\Users\laksh\Desktop\IJSE\came-ijse\Day 18
Database : C:\Users\laksh\Desktop\IJSE\came-ijse\Day 18\mlflow.db
Artifacts: C:\Users\laksh\Desktop\IJSE\came-ijse\Day 18\mlartifacts
Log      : C:\Users\laksh\Desktop\IJSE\came-ijse\Day 18\mlflow_server.log


## 4. Connect the notebook to the server

In [11]:
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("MLflow Quickstart")
print("MLflow Version:", mlflow.__version__)
print("MLflow Tracking URI:", mlflow.get_tracking_uri())
print(
    "MLflow Artifact URI (experiment):",
    mlflow.get_experiment_by_name("MLflow Quickstart").artifact_location,
)

MLflow Version: 3.16.1
MLflow Tracking URI: http://127.0.0.1:5000
MLflow Artifact URI (experiment): mlflow-artifacts:/1


## 5. Prepare the Iris Data

The Iris dataset has **150 flowers**, each described by 4 measurements, and 3 species to predict.

| Features (X) | Target (y) |
|---|---|
| sepal length, sepal width, petal length, petal width (cm) | `0` setosa · `1` versicolor · `2` virginica |

We keep 80 % of the rows to **train** the model and hold back 20 % to **test** it on flowers it has never seen.

In [ ]:
# ---------------------------------------------------------------------------
# Step 1: Load the Iris dataset
# as_frame=True gives a pandas DataFrame, so the columns keep their names.
# MLflow uses those names in the model's signature (shown in the UI).
# ---------------------------------------------------------------------------
iris = datasets.load_iris(as_frame=True)
X = iris.data  # 4 feature columns (the measurements)
y = iris.target  # 0, 1 or 2 (the species)

print("Features:", list(X.columns))
print("Classes :", iris.target_names.tolist())
print("Shape   :", X.shape)  # (rows, columns)

# ---------------------------------------------------------------------------
# Step 2: Split into train (80 %) and test (20 %) sets
# random_state=42 -> the same split every time, so results can be repeated.
# stratify=y      -> each set keeps the same share of each species.
# ---------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train rows: {len(X_train)} | Test rows: {len(X_test)}")

## 6. Train the Model and Log the Run to MLflow

Everything inside `with mlflow.start_run():` belongs to **one run**. A run is one training attempt, and it shows up as one row in the dashboard.

| What we log | MLflow call | Where it appears in the UI |
|---|---|---|
| Hyperparameters | `mlflow.log_params()` | Run → Overview → **Parameters** |
| Scores | `mlflow.log_metrics()` | Run → Overview → **Metrics** |
| Tag `Training Info` | `mlflow.set_tag()` | Run → Overview → **Tags** |
| The trained model | `mlflow.sklearn.log_model()` | Run → **Artifacts** / experiment → **Models** |

A **tag** is a free-text label. It does not change training. It only describes the run, so you can search and filter runs later (for example `tags."Training Info" LIKE '%iris%'`).

In [ ]:
# Turn off autologging so that ONLY what we log by hand is recorded
# (otherwise MLflow would log a second copy of everything automatically).
mlflow.sklearn.autolog(disable=True)

# ---------------------------------------------------------------------------
# Step 1: Choose the hyperparameters (settings fixed BEFORE training)
#   solver       -> the algorithm used to find the best weights
#   max_iter     -> the most optimisation steps it may take
#   random_state -> fixes randomness so the result is repeatable
# ---------------------------------------------------------------------------
params = {"solver": "lbfgs", "max_iter": 1000, "random_state": 8888}

# ---------------------------------------------------------------------------
# Step 2: Start an MLflow run. Everything logged inside this block is saved
#         under one run. run_name is the name shown in the dashboard.
# ---------------------------------------------------------------------------
with mlflow.start_run(run_name="iris-logistic-regression") as run:

    # Step 3: Log the hyperparameters -> Overview > Parameters
    mlflow.log_params(params)

    # Step 4: Create the model and train it on the training set only
    lr = LogisticRegression(**params)
    lr.fit(X_train, y_train)

    # Step 5: Predict on both sets
    #   train predictions -> how well it learned the data it saw
    #   test predictions  -> how well it works on NEW data (the score that matters)
    y_train_pred = lr.predict(X_train)
    y_test_pred = lr.predict(X_test)

    # Step 6: Score the predictions
    #   average="macro" -> work out the score for each of the 3 species,
    #                      then take the plain average
    metrics = {
        "train_accuracy": accuracy_score(y_train, y_train_pred),
        "test_accuracy": accuracy_score(y_test, y_test_pred),
        "precision": precision_score(y_test, y_test_pred, average="macro"),
        "recall": recall_score(y_test, y_test_pred, average="macro"),
        "f1_score": f1_score(y_test, y_test_pred, average="macro"),
    }

    # Step 7: Log the scores -> Overview > Metrics
    mlflow.log_metrics(metrics)

    # Step 8: Set the tag "Training Info" -> Overview > Tags
    #         A tag describes the run; it is handy for searching later.
    mlflow.set_tag("Training Info", "Basic LR model for iris data")

    # Step 9: Record the model's signature (input columns -> output type).
    #         MLflow uses it to check the input when the model is used later.
    signature = infer_signature(X_train, lr.predict(X_train))

    # Step 10: Save the trained model into MLflow (stored in mlartifacts/)
    #          input_example -> a few sample rows saved with the model
    model_info = mlflow.sklearn.log_model(
        sk_model=lr,
        name="iris_model",
        signature=signature,
        input_example=X_train.head(3),
    )

# The run ends automatically when the "with" block finishes.

# Step 11: Show what was logged
for name, value in metrics.items():
    print(f"{name:<15}: {value:.4f}")
print("Run ID   :", run.info.run_id)
print("Model URI:", model_info.model_uri)

## 7. View the Run in the MLflow Dashboard

Open **http://127.0.0.1:5000** in your browser (or click the link the next cell prints), then follow this path:

```
Experiments ─► MLflow Quickstart ─► iris-logistic-regression ─┬─► Overview
                                                              │     ├─ Parameters : solver, max_iter, random_state
                                                              │     ├─ Metrics    : train_accuracy, test_accuracy, precision, recall, f1_score
                                                              │     └─ Tags       : Training Info = "Basic LR model for iris data"
                                                              └─► Artifacts ─► iris_model (model files, signature, input example)
```

| To do this | Do this in the UI |
|---|---|
| See the tag on every run | On the experiment's runs table, click **Columns** and tick `Training Info` |
| Find runs by tag | Type `tags."Training Info" LIKE '%iris%'` in the search box |
| Compare runs | Tick two or more runs, then click **Compare** |

Every time you re-run section 6, a **new run** is added, so you can change `params` and compare the results side by side.

In [ ]:
# Build direct links to this run and its experiment in the dashboard
experiment_url = f"{TRACKING_URI}/#/experiments/{run.info.experiment_id}"
run_url = f"{experiment_url}/runs/{run.info.run_id}"

print("Experiment:", experiment_url)
print("This run  :", run_url)

# Read the run back from the server to confirm the tag was saved
saved_run = mlflow.get_run(run.info.run_id)
print("Training Info tag:", saved_run.data.tags.get("Training Info"))

## 8. Load the Logged Model Back and Predict

The model is now stored in MLflow, so we can load it using its **model URI** (no need to retrain). `mlflow.pyfunc.load_model()` gives a general model with a `.predict()` method, no matter which library trained it.

In [ ]:
# Step 1: Load the model from MLflow using the URI printed in section 6
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)

# Step 2: Predict on the test set (rows the model never trained on)
predictions = loaded_model.predict(X_test)

# Step 3: Put the actual and predicted species side by side to compare
result = X_test.copy()
result["actual_class"] = [iris.target_names[i] for i in y_test]
result["predicted_class"] = [iris.target_names[i] for i in predictions]
result.head(10)

## 9. Stop the MLflow Server

Run this only when you have finished with the dashboard. It stops the server **only if this kernel started it** (section 3). If the server was already running from an earlier session, it is left running.

In [ ]:
# `server` only exists if section 3 started the server in this kernel
if "server" in globals() and server.poll() is None:
    # /T also stops the worker processes the server started; /F forces it
    subprocess.run(
        ["taskkill", "/PID", str(server.pid), "/T", "/F"], capture_output=True
    )
    server_log.close()
    print("MLflow server stopped.")
else:
    print("This kernel did not start the server, so it was left running.")